In [12]:
# %pip install pandas openpyxl
# %pip install rdflib

import time
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd


# RMO (Rijksmuseum van Oudheden)

In [13]:

BASE_URL = "http://api.rmo.nl:17521/"
IMAGE_BASE = "https://www.rmo.nl/imageproxy/"
QUERIES = {
    "nehalennia": None,
    "colijnsplaat": ("Vindplaats", "colijnsplaat"),
}
PAGE_SIZE = 100
SEP = " | "

OUT_DIR = Path("out")
RAW_DIR = OUT_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

## Fetch
The API takes its parameters as a path (`action=get&command=search&...`), not a query string. Raw XML pages are cached to `out/raw/` so reruns don't hit the server.

In [9]:
def fetch_page(query, start, end, use_cache=True):
    cache = RAW_DIR / f"{query}_{start}-{end}.xml"
    if use_cache and cache.exists():
        return ET.fromstring(cache.read_bytes())
    params = f"action=get&command=search&query={urllib.parse.quote(query)}&range={start}-{end}&fields=*"
    with urllib.request.urlopen(BASE_URL + params, timeout=60) as resp:
        body = resp.read()
    cache.write_bytes(body)
    time.sleep(0.5)  # be polite
    return ET.fromstring(body)


def harvest(query):
    first = fetch_page(query, 1, PAGE_SIZE)
    total = int(first.findtext("request/count"))
    print(f"{total} hits for {query!r}")
    pages = [first]
    for start in range(PAGE_SIZE + 1, total + 1, PAGE_SIZE):
        pages.append(fetch_page(query, start, min(start + PAGE_SIZE - 1, total)))
    records = [rec for page in pages for rec in page.iter("record")]
    print(f"harvested {len(records)} records")
    return records


def matches(rec, field_filter):
    if field_filter is None:
        return True
    field, value = field_filter
    return any(value.lower() in (el.text or "").lower() for el in rec.findall(f"data/RMO_XML/{field}"))


records = {}  # recordnumber -> (record, [queries it matched])
for query, field_filter in QUERIES.items():
    kept = [rec for rec in harvest(query) if matches(rec, field_filter)]
    if field_filter:
        print(f"kept {len(kept)} with {field_filter[0]} containing {field_filter[1]!r}")
    for rec in kept:
        key = rec.findtext("data/RMO_XML/recordnumber")
        records.setdefault(key, (rec, []))[1].append(query)
print(f"{len(records)} unique records")

276 hits for 'nehalennia'
harvested 276 records
386 hits for 'colijnsplaat'
harvested 386 records
kept 386 with Vindplaats containing 'colijnsplaat'
411 unique records


## Flatten (for information specialists to link)

In [10]:
def image_url(filename):
    stem, _, ext = filename.rpartition(".")
    return f"{IMAGE_BASE}{ext.lower()}/{stem}"


# the API returns ~27 fields; keep only these, renamed {column: API field}
FIELDS = {"object": "Objectnaam", "periode": "Periode", "datum_van": "datebegin",
          "datum_tot": "dateend", "vindplaats": "Vindplaats", "beschrijving": "Beschrijving"}


def flatten(rec):
    data = rec.find("data/RMO_XML")
    row, images = {}, []
    for el in data:
        text = (el.text or "").strip()
        if el.tag == "image":
            images.append({"filename": text, "image_url": image_url(text), **el.attrib})
            continue
        row.setdefault(el.tag, []).append(text)
    row = {k: SEP.join(v) for k, v in row.items()}

    kept = {"pid": row.get("europeana_isShownAt")}  # the handle link
    kept.update({column: row.get(field) for column, field in FIELDS.items()})
    kept["image_url"] = images[0]["image_url"] if images else None
    for i in images:
        i["pid"] = kept["pid"]
    return kept, images


rows, image_rows = [], []
for rec, queries in records.values():
    row, imgs = flatten(rec)
    rows.append(row)
    image_rows.extend(imgs)

df = pd.DataFrame(rows)

df_images = pd.DataFrame(image_rows)
df_images = df_images[["pid"] + [c for c in df_images if c != "pid"]]

print(df.shape, df_images.shape)
df.head()

(411, 8) (1228, 5)


,pid,object,periode,datum_van,datum_tot,vindplaats,beschrijving,image_url
0,https://hdl.handle.net/21.12126/137816,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",Altaar met nis. Vooraan rechtsonder is een hoe...,https://www.rmo.nl/imageproxy/tiff/019732
1,https://hdl.handle.net/21.12126/137817,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...","Altaar met nis. Kalksteen. Nehalennia zittend,...",https://www.rmo.nl/imageproxy/tiff/019527
2,https://hdl.handle.net/21.12126/137818,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",Altaar. Kalksteen. Rechter- en linkerkant: boo...,https://www.rmo.nl/imageproxy/tiff/019423
3,https://hdl.handle.net/21.12126/137819,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",Altaar met nis. Kalksteen. Afwijkend type in z...,https://www.rmo.nl/imageproxy/tiff/015782
4,https://hdl.handle.net/21.12126/138005,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...","Altaar met nis, kalksteen. Nehalennia zittend....",https://www.rmo.nl/imageproxy/tiff/019737


## Verwerving (scraped)

The API has no acquisition field, so this reads `Verwerving` off the public object page, where it is
one of the `data-list-item` rows. The object id is the last segment of the handle, which is the same
number as the API's `recordnumber`.

Pages are cached to `out/pages/`, so only the first run hits the site — about 4 minutes for 411
records at one request every half second.

In [ ]:
import html
import re

DETAIL_URL = "https://www.rmo.nl/collectie/collectiezoeker/collectiestuk/?object={}"
PAGE_DIR = OUT_DIR / "pages"
PAGE_DIR.mkdir(parents=True, exist_ok=True)

LIST_ITEM = re.compile(r'<div class="data-list-item">\s*<label>(.*?)</label>(.*?)</div>', re.S)
HEARTBEAT = 10  # progress line every N records


def fetch_detail(object_id, use_cache=True):
    cache = PAGE_DIR / f"{object_id}.html"
    if use_cache and cache.exists():
        return cache.read_text(encoding="utf-8", errors="replace")
    # the site 403s urllib's default User-Agent, so identify the client
    request = urllib.request.Request(DETAIL_URL.format(object_id),
                                     headers={"User-Agent": "hackalod-harvester/1.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        page = response.read().decode("utf-8", errors="replace")
    cache.write_text(page, encoding="utf-8")
    time.sleep(0.5)  # be polite
    return page


def detail_fields(page):
    """label -> value for each <div class="data-list-item"> row on the object page."""
    fields = {}
    for label, value in LIST_ITEM.findall(page):
        text = " ".join(html.unescape(re.sub(r"<[^>]+>", " ", value)).split())
        fields.setdefault(" ".join(html.unescape(label).split()), text)
    return fields


total = len(df)
print(f"scraping {total} object pages, cached ones are reused", flush=True)

verwerving, downloaded, started = [], 0, time.time()
for n, pid in enumerate(df["pid"], start=1):
    object_id = pid.rsplit("/", 1)[-1]
    downloaded += not (PAGE_DIR / f"{object_id}.html").exists()
    verwerving.append(detail_fields(fetch_detail(object_id)).get("Verwerving"))

    if n % HEARTBEAT == 0 or n == total:
        elapsed = time.time() - started
        eta = (total - n) * elapsed / n
        print(f"  {n:>4}/{total}  {downloaded} downloaded, {n - downloaded} cached"
              f"  |  {elapsed:>4.0f}s elapsed, ~{eta:.0f}s to go", flush=True)

df["verwerving"] = verwerving

found = df["verwerving"].notna().sum()
print(f"\ndone in {time.time() - started:.0f}s: {found} of {total} records have verwerving", flush=True)
df["verwerving"].value_counts().head(10)

scraping 411 object pages, cached ones are reused
    10/411  5 downloaded, 5 cached  |    10s elapsed, ~404s to go
    20/411  15 downloaded, 5 cached  |    27s elapsed, ~522s to go
    30/411  25 downloaded, 5 cached  |    43s elapsed, ~547s to go


## Export

In [16]:
EXCEL_CELL_LIMIT = 32767
too_long = df.map(lambda v: isinstance(v, str) and len(v) > EXCEL_CELL_LIMIT)
if too_long.any().any():
    print("truncating over-long cells in:", list(df.columns[too_long.any()]))
    df = df.map(lambda v: v[:EXCEL_CELL_LIMIT] if isinstance(v, str) else v)



In [ ]:
LINK_COLUMNS = ["pid", "image_url"]


def make_links_clickable(ws, columns=LINK_COLUMNS):
    header = [c.value for c in ws[1]]
    for name in columns:
        if name not in header:
            continue
        col = header.index(name) + 1
        for (cell,) in ws.iter_rows(min_row=2, min_col=col, max_col=col):
            if cell.value:
                cell.hyperlink = cell.value
                cell.style = "Hyperlink"

out_file = OUT_DIR / f"rmo_{'_'.join(QUERIES)}.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as xw:
    df.to_excel(xw, sheet_name="records", index=False)
    df_images.to_excel(xw, sheet_name="images", index=False)
    for ws in xw.sheets.values():
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        make_links_clickable(ws)

## Collectie Zeeland

The PIDs are hardcoded since 3 turn up by searching the dump for "Nehalennia", and the 4th (the dolium) does not. its museum description mentions Nehalennia, but this export carries no free-text
description field, so no search of the file can find it. Due to the small amount we hardcoded for simplicity

In [18]:
import gzip

from rdflib import BNode, Graph, Literal, Namespace, URIRef
from rdflib.namespace import RDF, RDFS

CRM = Namespace("http://www.cidoc-crm.org/cidoc-crm/")
SDO = Namespace("https://schema.org/")

ZEELAND_URL = ("https://zoeken.collectiezeeland.nl/Atlantispubliek/data/set/"
               "Zeeuws+erfgoedplatform+-+Museum+objecten.nq.gz")
ZEELAND_TTL = OUT_DIR / "zeeland_nehalennia.ttl"
ZEELAND_XLSX = OUT_DIR / "zeeland_nehalennia.xlsx"

PIDS = [
    "https://hdl.handle.net/21.12128/326160703",  # Nehalennia
    "https://hdl.handle.net/21.12128/326163937",  # Nehalennia
    "https://hdl.handle.net/21.12128/327230837",  # Votiefaltaar (replica) voor de godin Nehalennia
    "https://hdl.handle.net/21.12128/327052914",  # Randfragment van een groot voorraadvat of dolium
]

# the server rejects urllib's default User-Agent with 403, so identify the client
request = urllib.request.Request(ZEELAND_URL, headers={"User-Agent": "hackalod-harvester/1.0"})
with urllib.request.urlopen(request, timeout=300) as response:
    nquads = gzip.decompress(response.read())

zg = Graph()
zg.parse(data=nquads, format="nt")  # .nq extension, but the file has no graph column
for prefix, ns in [("crm", CRM), ("schema", SDO), ("aat", "http://vocab.getty.edu/aat/")]:
    zg.bind(prefix, ns)
print(f"{len(nquads) / 1e6:.0f} MB, {len(zg)} triples")

34 MB, 318042 triples


In [ ]:
values = " ".join(f"<{pid}>" for pid in PIDS)
described = zg.query(f"""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
DESCRIBE ?root ?linked WHERE {{
  VALUES ?root {{ {values} }}
  OPTIONAL {{
    ?root (!rdf:type)* ?mid .
    ?mid ?lp ?linked .
    FILTER(isIRI(?linked) && ?lp != rdf:type)
    FILTER EXISTS {{ ?linked ?x ?y }}
  }}
}}""").graph
described.namespace_manager = zg.namespace_manager
described.serialize(ZEELAND_TTL, format="turtle")
print(f"wrote {len(described)} triples to {ZEELAND_TTL.resolve()}")

wrote 248 triples to /Users/Minh.Tran/Documents/rijks/cit_hackalod/2026-nehalennia/fetch/out/zeeland_nehalennia.ttl


## Flatten (for information specialists to link)

In [20]:
def label_of(node):
    """Literal -> its text; IRI or blank node -> its rdfs:label if the dump has one, else the IRI."""
    if node is None:
        return None
    if isinstance(node, Literal):
        return str(node)
    label = zg.value(node, RDFS.label)
    return str(label) if label else str(node)


def joined(values):
    return SEP.join(dict.fromkeys(v for v in values if v))


def flatten_object(pid):
    obj = URIRef(pid)

    numbers, title_variants = [], []
    for node in zg.objects(obj, CRM.P1_is_identified_by):
        content = zg.value(node, CRM.P190_has_symbolic_content)
        if (node, RDF.type, CRM.E33_E41_Linguistic_Appellation) in zg:
            title_variants.append(str(content))
        else:
            numbers.append(f"{label_of(zg.value(node, CRM.P2_has_type))}: {content}")

    materials, notes = [], []
    for node in zg.objects(obj, CRM.P45_consists_of):
        materials.append(label_of(zg.value(node, CRM.P2_has_type)))
        notes += [str(n) for n in zg.objects(node, CRM.P3_has_note)]

    sizes = [str(s) for part in zg.objects(obj, CRM.P46_is_composed_of)
             for s in zg.objects(part, SDO.size)]

    makers, places, begins, ends = [], [], [], []
    for production in zg.objects(obj, CRM.P108i_was_produced_by):
        makers += [label_of(a) for a in zg.objects(production, CRM.P14_carried_out_by)]
        places += [label_of(p) for p in zg.objects(production, CRM.P7_took_place_at)]
        for span in zg.objects(production, CRM["P4_has_time-span"]):  # hyphen, so no attribute access
            begins += [str(v)[:10] for v in zg.objects(span, CRM.P82a_begin_of_the_begin)]
            ends += [str(v)[:10] for v in zg.objects(span, CRM.P82b_end_of_the_end)]

    images = [str(i) for i in zg.objects(obj, SDO.image) if isinstance(i, Literal)]
    manifests = [str(url) for node in zg.objects(obj, SDO.image) if isinstance(node, BNode)
                 for url in zg.objects(node, SDO.contentUrl)]

    return {
        "objectnummer": str(zg.value(obj, SDO.identifier) or ""),
        "titel": label_of(obj),
        "pid": pid,
        "objecttype": joined(label_of(t) for t in zg.objects(obj, CRM.P2_has_type)),
        "materiaal": joined(materials),
        "materiaal_toelichting": joined(notes),
        "afmetingen": joined(sizes),
        "datering_begin": joined(begins),
        "datering_eind": joined(ends),
        "vervaardiger": joined(makers),
        "vindplaats": joined(places),
        "andere_nummers": joined(numbers),
        "titelvarianten": joined(title_variants),
        "iiif_manifest": joined(manifests),
        "afbeelding_count": len(images),
        "afbeelding_url": images[0] if images else None,
        "afbeeldingen": joined(images),
        "licentie": str(zg.value(obj, SDO.license) or ""),
    }


df_zeeland = pd.DataFrame([flatten_object(pid) for pid in PIDS])

with pd.ExcelWriter(ZEELAND_XLSX, engine="openpyxl") as xw:
    df_zeeland.to_excel(xw, sheet_name="objects", index=False)
    for ws in xw.sheets.values():
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        make_links_clickable(ws, ["pid", "iiif_manifest", "afbeelding_url"])

print(f"wrote {len(df_zeeland)} objects to {ZEELAND_XLSX.resolve()}")
df_zeeland[["objectnummer", "titel", "objecttype", "materiaal", "vervaardiger", "afbeelding_count"]]

wrote 4 objects to /Users/Minh.Tran/Documents/rijks/cit_hackalod/2026-nehalennia/fetch/out/zeeland_nehalennia.xlsx


,objectnummer,titel,objecttype,materiaal,vervaardiger,afbeelding_count
0,MDC0139,Nehalennia,beelden (CHT),,"Metsers, Guido (RKDartists)",1
1,MDC0225,Nehalennia,objecten (CHT),,Ineke de Brouwer,1
2,MCA00408,Votiefaltaar (replica) voor de godin Nehalennia,votiefaltaar,composiet,,4
3,MCA00113,Randfragment van een groot voorraadvat of dolium,dolium aardewerk:dolium groot formaat (ABR),aardewerk,,1
